# Модель SARIMA (пакет [`statsmodels`](https://www.statsmodels.org/stable/index.html))

In [ ]:
import numpy as np
import pandas as pd

from statsmodels.tsa.api import ARIMA
# альтернатива
# from statsmodels.tsa.api import SARIMAX
from statsmodels.graphics.tsaplots import plot_predict

import pandas_datareader.data as web

# настройки визуализации
import matplotlib.pyplot as plt

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

Загрузим из БД [`FRED`](https://fred.stlouisfed.org/) дневные данные по Gross Domestic Product () (Symbol [`NA000334Q`](https://fred.stlouisfed.org/series/NA000334Q)) с 1990-01-01 по 2025-12-31 (ряд `z`) и создадим датафрейм `y=log(z)`

In [ ]:
z = web.DataReader(name='NA000334Q', data_source='fred', start='1990-01-01', end='2025-12-31')
y = np.log(z)

y.plot()
plt.show()

## Подгонка SARIMA заданного порядка

Подгоним модель SARMA(1,0,1)(1,1,1,4) с линейным трендом для `y` (число сезонов выберем равным 4)

Спецификация

$$
	(1-\phi L)(1-\Phi L^4)(1-L^4) (y_t-\gamma t)=(1+\theta L)(1+\Theta L^4)u_t
$$

In [ ]:
# спецификация модели
mod = ARIMA(y, order=(1,0,1), seasonal_order=(1,1,1,4), trend='t', missing='drop')
# подгонка модели на данных
res = mod.fit()
# выводим результаты подгонки
res.summary(alpha=0.05)

In [ ]:
# # альтернативно
# # спецификация модели
# mod = SARIMAX(y, order=(2,1,1), trend='n', missing='drop')
# # подгонка модели на данных
# res = mod.fit()
# # выводим результаты подгонки
# res.summary(alpha=0.05)

## Прогнозирование

Построим прогноз по модели SARMA(1,1,1)(1,0,1,4) с линейным трендом для `y` (число сезонов выберем равным 4)

Спецификация

$$
	(1-\phi L)(1-\Phi L^4)(1-L) (y_t-\gamma t)=(1+\theta L)(1+\Theta L^4)u_t
$$

In [ ]:
# спецификация модели
mod = ARIMA(y, order=(1,1,1), seasonal_order=(1,0,1,4), trend='t', missing='drop')
# подгонка модели на данных
res = mod.fit()
# прогнозирование
forecasts = res.forecast(steps=10)
forecasts

визуализация прогноза

In [ ]:
plot_predict(res, start=len(y), end=len(y)+10, alpha=0.05)

plt.show()

In [ ]:
plt.plot(y.tail(30), label='y')
plt.plot(forecasts, label='y_pred')
plt.legend()

plt.show()